[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/06_anp_iot_palmor.ipynb)

# ANP, caso IoT/WSN Palmor (con interdependencia nueva)

Extiende el AHP de arriba. Elección de tecnología de comunicación (**LoRaWAN, GSM/GPRS, Sigfox, Zigbee**) para una red de sensores IoT/WSN de monitoreo agroclimático en Palmor, corregimiento de Ciénaga (Sierra Nevada de Santa Marta, Magdalena), zona con conectividad limitada verificada vía MinTIC. Los 4 criterios (Alcance de comunicación, Autonomía de batería, Infraestructura/cobertura comercial en Colombia, Madurez/viabilidad comercial del proveedor) salieron del panel de evidencia de la Sesión 1. Los valores técnicos de la matriz de decisión son reales, verificados vía WebSearch (datasheets SIMCom/DigiKey, The Things Network, Lauridsen et al. 2019 *Sensors*/MDPI, noticias de apagado de 2G en Colombia).

**Nota de alcance:** igual que ELECTRE/PROMETHEE arriba, esta interdependencia es una extensión nueva de este repositorio (en el curso real, el bloque ANP de S6 se queda **permanentemente** en el caso cacao, por ser un problema espacial continuo distinto al de elegir entre 4 alternativas discretas). La interdependencia que se propone aquí, razonada sobre hechos ya establecidos del caso (no inventada al azar):

- Sigfox y GSM/GPRS tienen **redes comerciales ya desplegadas** en Colombia (Infraestructura 5 y 3 respectivamente) → sus cifras de Alcance y Autonomía vienen de **despliegues reales en campo**, más confiables.
- LoRaWAN y Zigbee **no dependen de un operador**, exigen infraestructura propia (Infraestructura 2 ambas) → sus cifras de Alcance y Autonomía vienen sobre todo de **datasheet/laboratorio**, menos validadas en campo.

Esa es la retroalimentación (Alternativas → Criterios) que AHP no puede modelar y ANP sí: qué tanto confiar en cada criterio depende de qué tecnología se está evaluando.

In [1]:
import numpy as np
from pyDecision.algorithm import ahp_method

criterios = ["Alcance", "Autonomia", "Infraestructura", "Madurez"]
tecnologias = ["LoRaWAN", "GSM/GPRS", "Sigfox", "Zigbee"]

## Paso 1, bloque Criterios→Alternativas (ya conocido de AHP)

In [2]:
CtoA = {
    "LoRaWAN":  [0.2370, 0.5620, 0.0920, 0.3870],
    "GSM/GPRS": [0.2000, 0.0700, 0.2690, 0.1550],
    "Sigfox":   [0.5020, 0.1900, 0.5450, 0.0710],
    "Zigbee":   [0.0620, 0.1770, 0.0930, 0.3870],
}

## Paso 2, bloque Alternativas→Criterios (NUEVO en ANP)

Dos perfiles de validación en campo: Sigfox/GSM-GPRS (alta, red comercial real desplegada, pesos concentrados en Alcance/Autonomía porque SÍ hay campo real que los valide), LoRaWAN/Zigbee (baja, sin red de operador, el peso se concentra en Infraestructura/Madurez, los hechos comerciales que sí se pueden verificar sin medición de campo).

In [3]:
m_alta = np.array([
    [1,   1,   4,   5],
    [1,   1,   4,   5],
    [1/4, 1/4, 1,   2],
    [1/5, 1/5, 1/2, 1],
])
m_baja = np.array([
    [1, 1, 1/4, 1/5],
    [1, 1, 1/4, 1/5],
    [4, 4, 1,   1],
    [5, 5, 1,   1],
])
w_alta, cr_alta = ahp_method(m_alta, wd='m')
w_baja, cr_baja = ahp_method(m_baja, wd='m')
print("Perfil alta validación en campo:", np.round(w_alta, 4), "CR=", round(cr_alta, 4))
print("Perfil baja validación en campo:", np.round(w_baja, 4), "CR=", round(cr_baja, 4))

AtoC = {"Sigfox": w_alta, "GSM/GPRS": w_alta, "LoRaWAN": w_baja, "Zigbee": w_baja}

Perfil alta validación en campo: [0.4055 0.4055 0.1158 0.0732] CR= 0.0103
Perfil baja validación en campo: [0.0913 0.0913 0.386  0.4314] CR= 0.0023


## Paso 3, construir la supermatriz 8×8 (4 criterios + 4 tecnologías)

In [4]:
n = 8
W = np.zeros((n, n))
# filas/columnas 0-3 = criterios, 4-7 = tecnologías
for i, t in enumerate(tecnologias):
    W[4 + i, 0:4] = CtoA[t]        # Criterios -> Alternativas
    W[0:4, 4 + i] = AtoC[t]        # Alternativas -> Criterios

print("Suma de cada columna (debe ser 1.0):", np.round(W.sum(axis=0), 4))

Suma de cada columna (debe ser 1.0): [1.001 0.999 0.999 1.    1.    1.    1.    1.   ]


## Paso 4, supermatriz límite

La red es bipartita (2 clústeres, sin auto-influencia interna): las potencias pares e impares oscilan entre dos patrones, se promedian dos potencias consecutivas (Cesàro) para obtener la convergencia real, mismo tratamiento que el caso cacao.

In [5]:
W40 = np.linalg.matrix_power(W, 40)
W41 = np.linalg.matrix_power(W, 41)
W_limite = (W40 + W41) / 2

print("¿Todas las columnas de la supermatriz límite son iguales?",
      np.allclose(W_limite, W_limite[:, [0]] * np.ones((1, n)), atol=1e-3))

final = W_limite[:, 0]
final = final / final.sum()
for etiqueta, valor in zip(criterios + tecnologias, final):
    print(f"  {etiqueta:16s} {valor:.4f}")

¿Todas las columnas de la supermatriz límite son iguales? True
  Alcance          0.1243
  Autonomia        0.1243
  Infraestructura  0.1254
  Madurez          0.1261
  LoRaWAN          0.1596
  GSM/GPRS         0.0868
  Sigfox           0.1633
  Zigbee           0.0902


## Paso 5, ranking final (renormalizado solo entre tecnologías)

In [6]:
prioridad_tec = final[4:8]
prioridad_tec = prioridad_tec / prioridad_tec.sum()
print("Ranking final ANP:")
for t, p in sorted(zip(tecnologias, prioridad_tec), key=lambda x: -x[1]):
    print(f"  {t:10s} {p:.4f}")

Ranking final ANP:
  Sigfox     0.3266
  LoRaWAN    0.3193
  Zigbee     0.1804
  GSM/GPRS   0.1737


**Resultado:** 1º Sigfox (0.3266) · 2º LoRaWAN (0.3193) · 3º Zigbee (0.1804) · 4º GSM/GPRS (0.1737). A diferencia del caso cacao (donde ANP no cambiaba el orden de AHP), aquí la retroalimentación sí **invierte el 3er y 4º lugar** (Zigbee supera a GSM/GPRS, al revés que en AHP/TOPSIS/PROMETHEE) y **cierra mucho la distancia entre Sigfox y LoRaWAN** (de 40.1% vs. 26.3% en AHP puro a 32.7% vs. 31.9% aquí) — los pesos de criterios también se aplanan (de 16/25/48.8/10.2% a ~25% cada uno), porque la retroalimentación pesa igual a dos tecnologías por perfil. Es un resultado real, calculado, no ajustado para parecerse a otro método.